# DRAG-Net on AffectNet-7: original protocol

The original protocol reported in Table 5: the model is trained on the full official training set (283,901 images),
and the official validation set (3,500 images) is used both for checkpoint selection and early stopping
(patience 15, 50 epochs) and as the test set. All other settings are those of Table 4: AdamW (learning rate 3 × 10⁻⁴,
weight decay 1 × 10⁻⁴), cosine annealing (T_max = 50, η_min = 10⁻⁶), cross-entropy with label smoothing 0.1,
batch size 32, input 224 × 224 × 3.

Expected folders: `ROOT/train/<class>/*.jpg` and `ROOT/val/<class>/*.jpg` (seven class folders; images already
preprocessed as described in Section 2.1). Run the cells in order: model → seed → training and test → confusion matrix.
Outputs are written to `runs/affectnet_original/` (`training_log.csv`, `test_results.json`, `test_predictions.npz`,
`confusion_matrix_test.png`).


In [ ]:
## ============================================
# MODEL CREATION
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b2
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast, GradScaler

# -----------------------
# CBAM Block (Attention)
# -----------------------
class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super(CBAMBlock, self).__init__()
        # Channel Attention
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid_channel = nn.Sigmoid()
        # Spatial Attention
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        # Channel Attention
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        channel_out = self.sigmoid_channel(avg_out + max_out).view(b, c, 1, 1)
        x = x * channel_out
        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial_out = torch.cat([avg_out, max_out], dim=1)
        spatial_out = self.sigmoid_spatial(self.conv_spatial(spatial_out))
        x = x * spatial_out
        return x

# ---------------------------------------------------------
# Basic Conv Block
# ---------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel=3, dropout=0.3):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.dropout(x)
        return x


# ---------------------------------------------------------
# Residual Block (2 conv blocks + skip)
# ---------------------------------------------------------
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBlock(channels, channels)
        self.conv2 = ConvBlock(channels, channels)

    def forward(self, x):
        identity = x
        x = self.conv1(x)
        x = self.conv2(x)
        return x + identity


# ---------------------------------------------------------
# BlockTwoSkip (2 residual blocks + skip)
# ---------------------------------------------------------
class BlockTwoSkip(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.res1 = ResidualBlock(channels)
        self.res2 = ResidualBlock(channels)

    def forward(self, x):
        identity = x
        x = self.res1(x)
        x = self.res2(x)
        return x + identity

class GraphConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        # x: [B, N, C]
        # adj: [N, N]
        out = torch.matmul(adj, x)          # message passing
        out = self.linear(out)
        return F.relu(out)

def build_grid_adj(h, w, device):
    n = h * w
    adj = torch.zeros(n, n, device=device)

    for i in range(h):
        for j in range(w):
            idx = i * w + j
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < h and 0 <= nj < w:
                    nidx = ni * w + nj
                    adj[idx, nidx] = 1

    # self-loop + normalization
    adj += torch.eye(n, device=device)
    deg = adj.sum(dim=1, keepdim=True)
    adj = adj / deg
    return adj

# ---------------------------------------------------------
# Full Model with densenet121 Backbone
# ---------------------------------------------------------
class FERModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        # -------- DenseNet Backbone --------
        backbone = models.densenet121(
            weights=models.DenseNet121_Weights.IMAGENET1K_V1
        )
        self.backbone = backbone.features   # [B, 1024, 7, 7]
        in_channels = 1024


        # Feature blocks (كما هي)
        self.conv1a = ConvBlock(in_channels, 128)
        mid_channels = 196   # 128

        # -------------------
        # Feature extractor
        # -------------------
        self.conv1 = ConvBlock(in_channels, mid_channels, dropout=0.2)
        self.block1 = BlockTwoSkip(mid_channels)

        self.conv2 = ConvBlock(mid_channels, mid_channels, dropout=0.2)
        self.block2 = BlockTwoSkip(mid_channels)

        self.cbam = CBAMBlock(mid_channels)

        self.conv3 = ConvBlock(mid_channels, mid_channels, dropout=0.2)
        self.block3 = BlockTwoSkip(mid_channels)

        # self.gap = nn.AdaptiveAvgPool2d(1)
        # GNN
        
        self.gnn1 = GraphConv(mid_channels, mid_channels)
        self.gnn2 = GraphConv(mid_channels, mid_channels)
        
        self.gnn_fc = nn.Linear(mid_channels, mid_channels)
        # -------------------
        # Classifier
        # -------------------
        self.classifier = nn.Sequential(
            nn.Linear(mid_channels, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
    
        x = self.conv1(x)
        x = self.block1(x)
    
        x = self.conv2(x)
        x = self.block2(x)
    
        x = self.cbam(x)
        x = self.conv3(x)
        x = self.block3(x)
    
        # --------------------
        # CNN → Graph
        # --------------------
        B, C, H, W = x.shape
        nodes = x.view(B, C, H*W).permute(0, 2, 1)  # [B, N, C]
    
        adj = build_grid_adj(H, W, x.device)
    
        nodes = self.gnn1(nodes, adj)
        nodes = self.gnn2(nodes, adj)
    
        # Graph pooling
        gnn_feat = nodes.mean(dim=1)   # [B, C]
        gnn_feat = self.gnn_fc(gnn_feat)
    
        # --------------------
        # Final classifier
        # --------------------
        logits = self.classifier(gnn_feat)
        return logits


In [ ]:
def set_seed(seed=42):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [ ]:
import os, csv, json, time, copy
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Needs from the cells above: FERModel, set_seed
# ORIGINAL protocol (Table 5). Same settings as Table 4, but the model is trained on the full official training set
# and the official validation set is used both for checkpoint selection / early stopping and as the test set.

# ------------------- CONFIG (Table 4, AffectNet-7) -------------------
ROOT          = 'D/AffectNet'     # folder that holds train/ and val/  (set to your path)
TRAIN_DIR     = os.path.join(ROOT, 'train')   # official training set (283,901 images), one sub-folder per class
TEST_DIR      = os.path.join(ROOT, 'val')     # official validation set (3,500 images, 500 per class)
BATCH_SIZE    = 32                # Table 4: batch size 32
EPOCHS        = 50                # Table 4: 50 epochs
LR            = 3e-4              # Table 4: AdamW, learning rate 3 x 10^-4
WEIGHT_DECAY  = 1e-4              # Table 4: weight decay (AdamW) 1 x 10^-4
ETA_MIN       = 1e-6              # Table 4: cosine annealing, T_max = epochs, eta_min = 10^-6
LABEL_SMOOTHING = 0.1             # Table 4: cross-entropy, label smoothing 0.1
PATIENCE      = 15                # Table 4: early stopping with patience 15
USE_CLASS_WEIGHTS = False         # Section 2.1.5: no class re-weighting or re-sampling
CKPT_NAME     = 'best_affectnet_original.pth'
OUT_DIR       = 'runs/affectnet_original'
IGNORE_CLASSES = {'contempt'}     # AffectNet-7 does not use contempt (use {'7'} if the class folders are AffectNet label codes 0-7)
NUM_WORKERS   = 0 if os.name == 'nt' else 4   # Windows + Jupyter cannot pickle notebook-defined classes into worker processes

for d in (TRAIN_DIR, TEST_DIR):
    assert os.path.isdir(d), f"missing folder: {d}"
os.makedirs(OUT_DIR, exist_ok=True)
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------- TRANSFORMS (224 x 224 x 3; augmentation of Section 2.1.3) -------------------
norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),                                  # Section 2.1.3
    transforms.RandomRotation(15),                                           # Section 2.1.3
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),    # Section 2.1.3
    transforms.ToTensor(),
    norm,
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    norm,
])

# ------------------- DATASET class that skips ignored folders -------------------
class FilteredImageFolder(datasets.ImageFolder):
    def find_classes(self, directory):
        classes, _ = super().find_classes(directory)
        classes = [c for c in classes if c.lower() not in IGNORE_CLASSES]
        return classes, {c: i for i, c in enumerate(classes)}

# ------------------- DATASETS: full official training set; official validation set -------------------
train_dataset = FilteredImageFolder(TRAIN_DIR, transform=train_transform)   # all 283,901 training images
test_dataset  = FilteredImageFolder(TEST_DIR,  transform=eval_transform)    # official validation set (3,500)

assert train_dataset.class_to_idx == test_dataset.class_to_idx, "class folders differ between train / val"
emotion_labels = train_dataset.classes
num_classes = len(emotion_labels)
assert not any(c.lower() in IGNORE_CLASSES for c in emotion_labels), "ignored class still present!"
print("Classes:", emotion_labels, f"({num_classes} classes, ignored: {sorted(IGNORE_CLASSES)})")
print(f"Train={len(train_dataset)} | official val={len(test_dataset)}")
print("Train class counts:       ", np.bincount(train_dataset.targets, minlength=num_classes))
print("Official val class counts:", np.bincount(test_dataset.targets, minlength=num_classes))
for name, n, expected in [('official train', len(train_dataset), 283901), ('official val', len(test_dataset), 3500)]:
    if n != expected:
        print(f"WARNING: {name} has {n} images, the manuscript reports {expected}")

# ------------------- LOADERS -------------------
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          drop_last=True)          # avoids a 1-sample last batch (BatchNorm1d)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = test_loader                         # original protocol: the official validation set is monitored
train_targets = np.array(train_dataset.targets)

# ------------------- MODEL / LOSS / OPTIMIZER / SCHEDULER -------------------
if USE_CLASS_WEIGHTS:
    from sklearn.utils.class_weight import compute_class_weight
    w = compute_class_weight('balanced', classes=np.arange(num_classes), y=train_targets)
    weight = torch.tensor(w, dtype=torch.float32, device=device)
else:
    weight = None
criterion = nn.CrossEntropyLoss(weight=weight, label_smoothing=LABEL_SMOOTHING)

model = FERModel(num_classes=num_classes).to(device)        # feature width C (mid_channels) is set inside FERModel
width_C = model.conv1.conv.out_channels

# Parameters: conv1a is defined in FERModel.__init__ but never used in forward(), so it is excluded from the reported count
n_params_total = sum(p.numel() for p in model.parameters())
n_params = sum(p.numel() for n, p in model.named_parameters() if not n.startswith('conv1a.'))
try:                                                          # optional: pip install fvcore
    from fvcore.nn import FlopCountAnalysis
    model.eval()
    with torch.no_grad():
        gflops = FlopCountAnalysis(model, torch.zeros(1, 3, 224, 224, device=device)).total() / 1e9
except Exception:
    gflops = None
print(f"Width C = {width_C} | parameters used in forward(): {n_params / 1e6:.3f} M "
      f"(all defined, incl. unused conv1a: {n_params_total / 1e6:.3f} M)"
      + (f" | {gflops:.2f} GFLOPs per 224 x 224 image (fvcore, multiply-adds)" if gflops else ""))

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=ETA_MIN)

# ------------------- helpers -------------------
def run_epoch(loader, train):
    model.train(train)
    total_loss, ys, probs = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * x.size(0)
        ys.append(y.cpu()); probs.append(torch.softmax(logits, 1).detach().cpu())
    y = torch.cat(ys).numpy(); p = torch.cat(probs).numpy()
    return total_loss / len(y), accuracy_score(y, p.argmax(1)), y, p

# ------------------- TRAIN: checkpoint selection and early stopping on the official validation set -------------------
best_val_acc, wait = -1.0, 0
log_path = os.path.join(OUT_DIR, 'training_log.csv')
with open(log_path, 'w', newline='') as f:
    csv.writer(f).writerow(['epoch', 'lr', 'train_loss', 'train_acc', 'official_val_loss', 'official_val_acc', 'seconds'])
for epoch in range(1, EPOCHS + 1):
    t0 = time.time(); lr = optimizer.param_groups[0]['lr']
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, train=True)
    va_loss, va_acc, _, _ = run_epoch(val_loader, train=False)
    scheduler.step()                                              # cosine annealing, once per epoch
    with open(log_path, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, f'{lr:.2e}', f'{tr_loss:.4f}', f'{tr_acc:.4f}', f'{va_loss:.4f}', f'{va_acc:.4f}', f'{time.time()-t0:.0f}'])
    print(f"Epoch {epoch:3d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | official_val {va_loss:.4f} / {va_acc:.4f}")
    if va_acc > best_val_acc:
        best_val_acc, wait = va_acc, 0
        torch.save(model.state_dict(), CKPT_NAME)                 # selected checkpoint
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping after epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

# ------------------- TEST: official validation set, with the selected checkpoint -------------------
model.load_state_dict(torch.load(CKPT_NAME, map_location=device))
_, test_acc, y_true, y_prob = run_epoch(test_loader, train=False)
y_pred = y_prob.argmax(1)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
results = {'test_accuracy': float(test_acc), 'macro_precision': float(prec), 'macro_recall': float(rec),
           'macro_f1': float(f1), 'n_test_images': int(len(y_true)), 'classes': emotion_labels,
           'confusion_matrix': confusion_matrix(y_true, y_pred, labels=list(range(num_classes))).tolist(),
           'width_C': int(width_C), 'parameters_used': int(n_params), 'parameters_all_defined': int(n_params_total),
           'gflops': gflops, 'protocol': 'original',
           'n_train': len(train_dataset)}
json.dump(results, open(os.path.join(OUT_DIR, 'test_results.json'), 'w'), indent=2)
np.savez(os.path.join(OUT_DIR, 'test_predictions.npz'), y_true=y_true, y_prob=y_prob, classes=np.array(emotion_labels))
print(f"\nAffectNet-7 TEST, original protocol ({len(y_true)} images): accuracy {100*test_acc:.2f}% | macro F1 {100*f1:.2f}%")


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# -------------------
# Evaluation Function
# -------------------
def evaluate_model(model, loader, device, emotion_labels, save_path=None):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Accuracy
    accuracy = 100 * (torch.tensor(all_preds) == torch.tensor(all_labels)).sum().item() / len(all_labels)
    print(f"✅ Test Accuracy: {accuracy:.2f}%")

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(emotion_labels))))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=emotion_labels,
                yticklabels=emotion_labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

    # Classification Report
    print("\n📊 Classification Report:")
    print(classification_report(all_labels, all_preds, labels=list(range(len(emotion_labels))),
                                target_names=emotion_labels, digits=4, zero_division=0))
    return cm


In [ ]:
# ------------------- CONFUSION MATRIX + CLASSIFICATION REPORT (TEST split, same selected checkpoint) -------------------
best_model = FERModel(num_classes=num_classes).to(device)
best_model.load_state_dict(torch.load(CKPT_NAME, map_location=device))
evaluate_model(best_model, test_loader, device, emotion_labels,
               save_path=os.path.join(OUT_DIR, 'confusion_matrix_test.png'))
